In [1]:
# !pip install emoji

In [2]:
import numpy as np
import pandas as pd

In [3]:
train = pd.read_csv('dataset/train_emoji.csv',header=None)
test = pd.read_csv('dataset/test_emoji.csv',header=None)

In [4]:
train.head()

,0,1,2,3
0,never talk to me again,3,NaN,NaN
1,I am proud of your achievements,2,NaN,NaN
2,It is the worst day in my life,3,NaN,NaN
3,Miss you so much,0,NaN,[0]
4,food is life,4,NaN,NaN


In [5]:
import emoji as emoji

In [6]:
#emoji.EMOJI_ALIAS_UNICODE

In [7]:
emoji_dictionary = {"0": "\u2764\uFE0F",    # :heart: prints a black instead of red heart depending on the font
                    "1": ":baseball:",
                    "2": ":beaming_face_with_smiling_eyes:",
                    "3": ":downcast_face_with_sweat:",
                    "4": ":fork_and_knife:",
                   }

In [8]:
emoji.emojize(":fork_and_knife:")

'🍴'

In [9]:
for e in emoji_dictionary.values():
    print(emoji.emojize(e))

❤️
⚾
😁
😓
🍴


In [10]:
data = train.values
for i in range(10):
    print(data[i][0],emoji.emojize(emoji_dictionary[str(data[i][1])]))

never talk to me again 😓
I am proud of your achievements 😁
It is the worst day in my life 😓
Miss you so much ❤️
food is life 🍴
I love you mum ❤️
Stop saying bullshit 😓
congratulations on your acceptance 😁
The assignment is too long  😓
I want to go play ⚾


## #Code to be Written in django applicattion

In [11]:
import h5py
from keras.utils import to_categorical


In [12]:
XT = train[0]
Xt = test[0]

YT = to_categorical(train[1])
Yt = to_categorical(test[1])


print(XT.shape)
print(Xt.shape)
print(YT.shape)
print(Yt.shape)

(132,)
(56,)
(132, 5)
(56, 5)


In [13]:
from pathlib import Path

for path in Path(".").rglob("*"):
    if "glove" in path.name.lower():
        print(path)


glove.6B.50d.txt


In [14]:
embeddings = {}
with open('glove.6B.50d.txt',encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coeffs = np.asarray(values[1:],dtype='float32')
        
        #print(word)
        #print(coeffs)
        embeddings[word] = coeffs

In [15]:
def getOutputEmbeddings(X):
    
    embedding_matrix_output = np.zeros((X.shape[0],10,50))
    for ix in range(X.shape[0]):
        X[ix] = X[ix].split()
        for jx in range(len(X[ix])):
            embedding_matrix_output[ix][jx] = embeddings[X[ix][jx].lower()]
            
    return embedding_matrix_output

In [16]:
XT[1]

'I am proud of your achievements'

In [17]:
emb_XT = getOutputEmbeddings(XT)
emb_Xt = getOutputEmbeddings(Xt)

C:\Users\AdityaUtkarsh\AppData\Local\Temp\ipykernel_7288\2176504226.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[ix] = X[ix].split()
C:\Users\AdityaUtkarsh\AppData\Local\Temp\ipykernel_7288\2176504226.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[ix] = X[ix].split()
C:\Users\AdityaUtkarsh\AppData\Local\Temp\ipykernel_7288\2176504226.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[ix] = X[ix].split()
C:\Users\Adit

In [18]:
print(emb_XT.shape)
print(emb_Xt.shape)


(132, 10, 50)
(56, 10, 50)


In [19]:
from keras.layers import LSTM, Dropout, Dense, Activation
from keras.models import Sequential

In [20]:
model = Sequential()
model.add(LSTM(64,input_shape=(10,50),return_sequences=True))
model.add(Dropout(0.4))
model.add(LSTM(64,input_shape=(10,50)))
model.add(Dropout(0.3))
model.add(Dense(5))
model.add(Activation('softmax'))
model.summary()

C:\Users\AdityaUtkarsh\.conda\envs\env\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 10, 64)              │          29,440 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 10, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 64)                  │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 5)                   │             325 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation (Activation)              │ (None, 5)                   │               0 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 62,789 (245.27 KB)

 Trainable params: 62,789 (245.27 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['acc'])
model.fit(emb_XT,YT,batch_size=32,epochs=40,shuffle=True,validation_split=0.1)

Epoch 1/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 12s 721ms/step - acc: 0.2627 - loss: 1.6020 - val_acc: 0.0714 - val_loss: 1.6347
Epoch 2/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - acc: 0.2288 - loss: 1.5565 - val_acc: 0.2143 - val_loss: 1.6367
Epoch 3/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - acc: 0.3729 - loss: 1.4951 - val_acc: 0.2857 - val_loss: 1.6309
Epoch 4/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - acc: 0.3475 - loss: 1.4678 - val_acc: 0.2857 - val_loss: 1.5949
Epoch 5/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - acc: 0.3898 - loss: 1.4291 - val_acc: 0.2857 - val_loss: 1.5670
Epoch 6/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - acc: 0.4746 - loss: 1.3614 - val_acc: 0.1429 - val_loss: 1.5381
Epoch 7/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - acc: 0.5254 - loss: 1.2901 - val_acc: 0.2143 - val_loss: 1.4678
Epoch 8/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - acc: 0.5678 - loss: 1.1954 - val_acc: 0.2857 - val_loss: 1.3935
Epoch 9/40
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - acc: 0.6186 - loss: 1.095

In [22]:
model.evaluate(emb_Xt,Yt)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - acc: 0.6250 - loss: 1.4469


[1.4469187259674072, 0.625]

In [24]:
pred = np.argmax(model.predict(emb_Xt), axis=1)


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 997ms/step


In [25]:
for i in range(30):
    print(' '.join(Xt[i]))
    print(emoji.emojize(emoji_dictionary[str(np.argmax(Yt[i]))]))
    print(emoji.emojize(emoji_dictionary[str(pred[i])]))

I want to eat
🍴
🍴
he did not answer
😓
😓
he got a raise
😁
❤️
she got me a present
❤️
❤️
ha ha ha it was so funny
😁
😁
he is a good friend
❤️
😁
I am upset
❤️
😓
We had such a lovely dinner tonight
❤️
😁
where is the food
🍴
🍴
Stop making this joke ha ha ha
😁
😁
where is the ball
⚾
⚾
work is hard
😓
😁
This girl is messing with me
😓
❤️
are you serious ha ha
😁
🍴
Let us go play baseball
⚾
⚾
This stupid grader is not working
😓
😓
work is horrible
😓
😓
Congratulation for having a baby
😁
😁
stop messing around
😓
😓
any suggestions for dinner
🍴
😁
I love taking breaks
❤️
❤️
you brighten my day
😁
❤️
I boiled rice
🍴
🍴
she is a bully
😓
❤️
Why are you feeling bad
😓
😓
I am upset
😓
😓
I worked during my birthday
😓
😁
My grandmother is the love of my life
❤️
❤️
enjoy your break
😁
🍴
valentine day is near
❤️
😁


In [28]:
with open("model.json", "w") as file:
    file.write(model.to_json())

model.save_weights("model.weights.h5")


In [29]:
from keras.models import model_from_json

In [30]:
with open("model.json", "r") as file:
    model = model_from_json(file.read())
model.load_weights("model.h5")

In [31]:
test_str = "Hello how are you"
X = pd.Series([test_str])

In [32]:
emb_X = getOutputEmbeddings(X)

In [35]:
import numpy as np

p = np.argmax(model.predict(emb_X), axis=1)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 893ms/step


In [36]:
p = np.argmax(model.predict(emb_X, verbose=0), axis=1)

In [37]:
print(' '.join(X[0]))
print(emoji.emojize(emoji_dictionary[str(p[0])]))

Hello how are you
😁
